# Importing the necessary libraries

In [ ]:
!pip install pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 31.5 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import pydicom
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Dataset, random_split

# Dataset from RSNA Pneumonia Detection Challenge
https://www.kaggle.com/c/rsna-pneumonia-detection-challenge/overview

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/dataset_rsna/train_labels.csv")

# Ambil satu label per patientId
df_labels = df.drop_duplicates(subset="patientId")[["patientId", "Target"]]
df_labels["class"] = df_labels["Target"].map({0: "Normal", 1: "Pneumonia"})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Menentukan direktori output dan membuat subfolder untuk setiap kelas.
Ini bertujuan untuk menyimpan gambar hasil konversi dari format DICOM ke JPG
sesuai dengan labelnya (Normal atau Pneumonia).
Lokasi direktori output untuk menyimpan hasil konversi

In [ ]:
# Lokasi direktori output untuk menyimpan hasil konversi
output_dir = "/content/rsna_jpg_dataset"

# Membuat folder Normal dan Pneumonia jika belum ada
os.makedirs(os.path.join(output_dir, "Normal"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "Pneumonia"), exist_ok=True)

 Bagian ini melakukan konversi gambar DICOM menjadi format JPG (PNG)
 dan menyimpannya ke Google Drive. Proses ini meliputi:
 1. Mount Google Drive agar bisa diakses.
 2. Mendefinisikan lokasi folder input (DICOM) dan output (JPG/PNG).
 3. Membaca file CSV untuk mendapatkan label pasien.
 4. Melakukan iterasi melalui setiap pasien:
    - Membaca file DICOM.
    - Mengkonversi data pixel menjadi array gambar.
    - Menormalkan nilai pixel.
    - Menyimpan gambar hasil konversi ke folder yang sesuai (Normal/Pneumonia)
      dalam format PNG.
    - Menangani potensi error saat konversi.

In [ ]:
import cv2
from tqdm import tqdm

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Path DICOM & CSV
dicom_folder = "/content/drive/MyDrive/dataset_rsna/train_images"
csv_path = "/content/drive/MyDrive/dataset_rsna/train_labels.csv"

# Folder output (Google Drive, aman & permanen)
output_folder = "/content/drive/MyDrive/rsna_dataset_jpg"
os.makedirs(os.path.join(output_folder, "Normal"), exist_ok=True)
os.makedirs(os.path.join(output_folder, "Pneumonia"), exist_ok=True)

# Baca CSV dan siapkan label unik
df = pd.read_csv(csv_path)
df_labels = df.drop_duplicates(subset="patientId")[["patientId", "Target"]]
df_labels["class"] = df_labels["Target"].map({0: "Normal", 1: "Pneumonia"})

# Loop untuk konversi semua gambar DICOM ke PNG
for idx, row in tqdm(df_labels.iterrows(), total=len(df_labels)):
    pid = row["patientId"]
    label = row["class"]
    dicom_path = os.path.join(dicom_folder, f"{pid}.dcm")

    try:
        dcm = pydicom.dcmread(dicom_path)
        img = dcm.pixel_array

        # Normalisasi pixel dari float → 0-255
        img = cv2.convertScaleAbs(img, alpha=(255.0 / img.max()))

        save_path = os.path.join(output_folder, label, f"{pid}.png")
        cv2.imwrite(save_path, img)
    except Exception as e:
        print(f"Gagal konversi {pid}: {e}")

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/dataset_rsna/train_labels.csv")
df_labels = df.drop_duplicates(subset="patientId")[["patientId", "Target"]]
df_labels["class"] = df_labels["Target"].map({0: "Normal", 1: "Pneumonia"})

print(df_labels["class"].value_counts())

class
Normal       20672
Pneumonia     6012
Name: count, dtype: int64


## Augmentation Data (Dataset RSNA)

In [ ]:
# Dataset RSNA
# Augmentasi untuk training

transform_rsna = transforms.Compose([
    transforms.Resize((224, 224)),                           # Standarisasi ukuran
    transforms.RandomAffine(degrees=2.86,                    # ±0.05 radians ≈ ±2.86 degrees
                            translate=(0.1, 0.1),            # Translasi ±10%
                            scale=(0.8, 1.2)),               # Skala ±20%
    transforms.RandomHorizontalFlip(),                       # Flipping horizontal
    transforms.ColorJitter(brightness=0.2, contrast=0.2),    # Tambah/kurangi brightness/kontras
    transforms.Grayscale(num_output_channels=3),             # Pastikan output RGB 3 channel
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])                       # Normalisasi karena grayscale
])


# Untuk validasi (tanpa augmentasi)
transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])


# Load dataset dari folder yang sudah kamu simpan di Google Drive
full_dataset = datasets.ImageFolder(root='/content/drive/MyDrive/rsna_dataset_jpg', transform=transform_rsna)

## Dataset dan Data Loader (RSNA)

In [ ]:
# Hitung jumlah data
total_size = len(full_dataset)
val_size = int(0.05 * total_size)
train_size = total_size - val_size

# Random split
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

In [ ]:
from torch.utils.data import DataLoader

# Override transform untuk validasi
val_dataset.dataset.transform = transform_val

# Buat DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

# Gabungkan untuk train_model
dataloaders = {
    'train': train_loader,
    'val': val_loader
}

In [ ]:
print(f"Jumlah data total: {total_size}")
print(f"Jumlah data training: {len(train_dataset)}")
print(f"Jumlah data validasi: {len(val_dataset)}")

Jumlah data total: 26684
Jumlah data training: 25350
Jumlah data validasi: 1334


## Inisialisasi Model

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Pre-trained CNN
googlenet = models.googlenet(pretrained=True)
resnet18 = models.resnet18(pretrained=True)
densenet121 = models.densenet121(pretrained=True)

# Modifikasi output layer jadi 2 kelas
googlenet.fc = nn.Linear(googlenet.fc.in_features, 2)
resnet18.fc = nn.Linear(resnet18.fc.in_features, 2)
densenet121.classifier = nn.Linear(densenet121.classifier.in_features, 2)

# Pindahkan ke GPU jika tersedia
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
googlenet = googlenet.to(device)
resnet18 = resnet18.to(device)
densenet121 = densenet121.to(device)

## Setup Loss, Optimizer, Scheduler
Menyiapkan fungsi loss (mengukur error), optimizer (algoritma untuk belajar), dan scheduler learning rate (mengatur kecepatan belajar) untuk pelatihan model.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer_googlenet = optim.Adam(googlenet.parameters(), lr=0.001)
optimizer_resnet18 = optim.Adam(resnet18.parameters(), lr=0.001)
optimizer_densenet121 = optim.Adam(densenet121.parameters(), lr=0.001)

scheduler_googlenet = lr_scheduler.StepLR(optimizer_googlenet, step_size=5, gamma=0.1)
scheduler_resnet18 = lr_scheduler.StepLR(optimizer_resnet18, step_size=5, gamma=0.1)
scheduler_densenet121 = lr_scheduler.StepLR(optimizer_densenet121, step_size=5, gamma=0.1)

## Training Model (Dataset RSNA)

In [ ]:
def train_model(model, dataloaders, criterion, optimizer, scheduler,
                num_epochs=25, patience=5, save_path='best_model.pth'):
    model.to(device)
    best_acc = 0.0
    best_model_wts = None
    wait = 0  # Untuk menghitung berapa kali tidak membaik

    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 20)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            # Simpan ke history
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())

                # Cek apakah val_acc membaik
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = model.state_dict()
                    torch.save(best_model_wts, save_path)
                    print(f"✅ Best model saved (val_acc improved to {best_acc:.4f})")
                    wait = 0
                else:
                    wait += 1
                    if wait >= patience:
                        print(f"\n⏹️ Early stopping triggered. No improvement in {patience} epochs.")
                        print(f"📁 Best model saved at: {save_path}")
                        model.load_state_dict(best_model_wts)
                        return model, history

        print(f"{phase.capitalize()} Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")

    # Final checkpoint jika tidak early stop
    if best_model_wts:
        model.load_state_dict(best_model_wts)
        torch.save(best_model_wts, save_path)
        print(f"\n📁 Final model saved at: {save_path} (val_acc = {best_acc:.4f})")

    return model, history

## Memanggil model untuk dilatih di dataset RSNA

In [ ]:
# Train GoogLeNet
googlenet_trained, history_googlenet = train_model(
    googlenet, dataloaders, criterion, optimizer_googlenet, scheduler_googlenet, num_epochs=25)

# Train ResNet-18
resnet18_trained, history_resnet18 = train_model(
    resnet18, dataloaders, criterion, optimizer_resnet18, scheduler_resnet18, num_epochs=25)

# Train DenseNet-121
densenet121_trained, history_densenet121 = train_model(
    densenet121, dataloaders, criterion, optimizer_densenet121, scheduler_densenet121, num_epochs=25)


Epoch 1/25
--------------------
✅ Best model saved (val_acc improved to 0.8291)
Val Loss: 0.3807, Acc: 0.8291

Epoch 2/25
--------------------
✅ Best model saved (val_acc improved to 0.8321)
Val Loss: 0.4041, Acc: 0.8321

Epoch 3/25
--------------------
Val Loss: 0.4321, Acc: 0.8313

Epoch 4/25
--------------------
Val Loss: 0.4948, Acc: 0.8238

Epoch 5/25
--------------------
Val Loss: 0.5897, Acc: 0.8178

Epoch 6/25
--------------------
Val Loss: 0.6534, Acc: 0.8118

Epoch 7/25
--------------------

⏹️ Early stopping triggered. No improvement in 5 epochs.
📁 Best model saved at: best_model.pth

Epoch 1/25
--------------------
✅ Best model saved (val_acc improved to 0.8118)
Val Loss: 0.5106, Acc: 0.8118

Epoch 2/25
--------------------
✅ Best model saved (val_acc improved to 0.8126)
Val Loss: 0.6752, Acc: 0.8126

Epoch 3/25
--------------------
Val Loss: 0.8833, Acc: 0.8096

Epoch 4/25
--------------------
Val Loss: 1.0084, Acc: 0.8013

Epoch 5/25
--------------------
Val Loss: 1.1196

## Menyimpan Hasil dari Training Model diatas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Ganti nama file sesuai model
torch.save(googlenet_trained.state_dict(), "/content/drive/MyDrive/model_rsna_v2/model_rsna_googlenet.pth")
torch.save(resnet18_trained.state_dict(), "/content/drive/MyDrive/model_rsna_v2/model_rsna_resnet18.pth")
torch.save(densenet121_trained.state_dict(), "/content/drive/MyDrive/model_rsna_v2/model_rsna_densenet121.pth")